## Импорты и параметры

### Привязываем Гугл-диск

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Импортируем библиотеки

In [ ]:
!pip install python-docx
!pip install -U transformers
!pip install -q trl bert-score openai
!pip install trl==0.11.3
!pip install -q accelerate>=1.8.0
!pip install -q bitsandbytes>=0.46.1
!pip uninstall pyarrow datasets -y
!pip install -q datasets

import time
import random
import re
import pandas as pd
from docx import Document
import numpy as np
from datasets import Dataset
import os

import torch
from trl import AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from bert_score import BERTScorer
import openai
from tqdm import tqdm

import bert_score
import transformers
import json
import matplotlib.pyplot as plt
import glob
from peft import PeftModel

torch.cuda.empty_cache()
import gc
gc.collect()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 107.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.3 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 1.4.0
    Uninstalling trl-1.4.0:
      Successfully uninstalled trl-1.4.0
Found existing installation: pyarrow 24.0.0

229

### Сохраняем рандом сид

In [ ]:
def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

set_random_seed()

### Сохраняем важные параметры

In [ ]:
USE_LLM_REWARD = True
OPENROUTER_API_KEY = ""
JUDGE_MODEL = "openai/gpt-oss-120b:free"
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
HYPOTHESIS = 'base'  # base / h1 / h2 / h3

### Загружаем модель

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

## Код для подсчета скоров

In [ ]:
bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False, device='cuda')

if USE_LLM_REWARD:
    openrouter_client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY
    )
last_llm_score = 0.0

llm_cache = {}

def get_llm_score(generated_text, source_text, topic):
    cache_key = (generated_text, source_text, topic)
    if cache_key in llm_cache:
        return llm_cache[cache_key]

    if not USE_LLM_REWARD:
        return None

    model = JUDGE_MODEL

    prompt = f"""Ты - социолог-эксперт в открытом кодировании интервью.
Тема интервью: {topic}
Текст интервью: {source_text}

Сгенерированные коды и цитаты (в формате <код>цитата</код>):
{generated_text}

Оцени результат по трём критериям (каждый от 0 до 1):
1. Релевантность - насколько код соответствует содержанию фрагмента интервью с учётом темы.
2. Когерентность - насколько формулировка кода грамматически корректна и стилистически приемлема.
3. Теоретический инсайт - степень соответствия кода концептуальным ожиданиям.

Ответ дай строго в формате: число, число, число (например: 0.85, 0.90, 0.75). Не пиши никаких пояснений."""

    max_retries = 5
    base_delay = 1.0
    max_delay = 30.0

    for attempt in range(max_retries):
        try:
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=50
            )

            if response and hasattr(response, 'choices') and response.choices:
                content = response.choices[0].message.content
                if content:

                    numbers = re.findall(r"(\d+\.?\d*)", content)
                    if len(numbers) >= 3:
                        try:
                            scores = [float(n) for n in numbers[:3]]
                            mean_score = np.mean(scores)
                            mean_score = max(0.0, min(1.0, mean_score))
                            llm_cache[cache_key] = mean_score
                            return mean_score
                        except ValueError:
                            print(f"LLM warning: non-numeric numbers: {numbers}", flush=True)
                    else:
                        print(f"LLM warning: expected 3 numbers, got {len(numbers)}. Content: {content[:200]}", flush=True)
                else:
                    print("LLM warning: empty content", flush=True)
            else:
                print("LLM warning: invalid response structure", flush=True)

            return None

        except Exception as e:
            is_retryable = False
            if hasattr(e, 'status_code'):
                if e.status_code == 429 or (500 <= e.status_code < 600):
                    is_retryable = True
            elif '429' in str(e) or 'rate limit' in str(e).lower():
                is_retryable = True

            if not is_retryable or attempt == max_retries - 1:
                print(f"LLM error (final): {type(e).__name__}: {e}", flush=True)
                return None
            delay = min(base_delay * (2 ** attempt), max_delay)
            jitter = random.uniform(0, 0.1 * delay)
            wait_time = delay + jitter
            print(f"LLM error (retryable): {e}. Retrying in {wait_time:.2f}s... (attempt {attempt+1}/{max_retries})", flush=True)
            time.sleep(wait_time)

    return None

def compute_reward(generated, target, source_text, topic, use_llm=True):
    global last_llm_score
    _, _, bert_f1 = bertscorer.score([generated], [target])
    bert_reward = bert_f1.item()
    if use_llm:
        new_llm = get_llm_score(generated, source_text, topic)
        if new_llm is not None:
            last_llm_score = new_llm
        llm_reward = last_llm_score
    else:
        llm_reward = 0.0
    return bert_reward + llm_reward, bert_reward, llm_reward

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Создание промпта для Qwen2.5

### Базовый промпт

In [ ]:
def build_prompt_base(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

### Первая гипотеза (системный промпт на русском языке)

In [ ]:
def build_prompt_h1(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""Ты эксперт в анализу интервью. Твоя задача -- разметить тематические коды и выделить соответствующие цитаты

Формат вывода (состоит из нескольких общих кодов, каждый сопровождается цитатами и конкретными кодами):
**Общий код 1: <сгенерируй общий код 1>**
"<текст цитаты>" - **<сгенерируй конкретный код 1> (Конкретный код)**
"<текст цитаты>" - **<сгенерируй конкретный код 2> (Конкретный код)**
**Общий код 2: <сгенерируй сбщий код 2>**
"<текст цитаты>" - **<сгенерируй конкретный код> (Конкретный код)**
**Общий код <номер общего кода>: <сгенерируй общий код>**
"<текст цитаты>" - **<сгенерируй конкретный код> (Конкретный код)**
и так далее. Ты выбираешь количество общих и конктерных кодов

Теперь закодируй интервью согласно плану. Важно: ответ должен содержать только коды и цитаты, без ненужных слов и повторений.
Цитаты должны быть строго из текста. Дай ответ на русском языке
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <номер общего кода>: <сгенерируй общий код>**
"<текст цитаты>" - **<сгенерируй конкретный код> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

### Вторая гипотеза (chain of thought)

In [ ]:
def build_prompt_h2(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

### Третья гипотеза (примеры темы и кодов из датасета)

In [ ]:
def build_prompt_h3(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

## Чтение тестовых данных из файла и добавление промпта

In [ ]:
import pandas as pd
test_df = pd.read_csv('test_data.csv')
test_dataset = Dataset.from_pandas(test_df)

### Функция для добавления промпта

In [ ]:
def prepare_dataset(hf_dataset, tokenizer):
    queries, targets = [], []
    topics, transcripts = [], []
    for ex in hf_dataset:
        if HYPOTHESIS == 'base':
            prompt_func = build_prompt_base
        elif HYPOTHESIS == 'h1':
            prompt_func = build_prompt_h1
        elif HYPOTHESIS == 'h2':
            prompt_func = build_prompt_h2
        else:
            prompt_func = build_prompt_h3

        queries.append(prompt_func(ex, tokenizer))
        targets.append(ex['coding'])
        topics.append(ex['topic'])
        transcripts.append(ex['transcript'])
    return Dataset.from_dict(
        {"query": queries, "coding": targets, "topic": topics, "transcript": transcripts}
    )

In [ ]:
test_dataset = prepare_dataset(test_dataset, tokenizer)

## Настройка генерации и запуск модели

### Функция генерации

In [ ]:
def clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

def generate_code(prompt, max_new_tokens=1024, num_beams=2):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3600).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

### Инференс и сбор метрик

In [ ]:
device = next(model.parameters()).device
predictions = []
all_rewards_total = []
all_rewards_bert = []
all_rewards_llm = []

clear_cuda_cache()

for idx in tqdm(range(len(test_dataset)), desc="Evaluating"):
    batch_item = test_dataset[idx]
    query = batch_item["query"]
    target = batch_item["coding"]
    src = batch_item["transcript"]
    topic = batch_item["topic"]

    pred = generate_code(query)

    total_reward, bert_reward, llm_reward = compute_reward(pred, target, src, topic, use_llm=USE_LLM_REWARD)

    print(f'BERTScore: {bert_reward}; LLM score: {llm_reward}')

    predictions.append({
        "query": query,
        "prediction": pred,
        "target": target,
        "source_text": src,
        "topic": topic,
        "reward_total": total_reward,
        "reward_bert": bert_reward,
        "reward_llm": llm_reward
    })

pred_df = pd.DataFrame(predictions)

print("\n=== Test Metrics ===")

print(f'\nGeneration strategy: Beam Search')
print(f"Average Total Reward: {np.mean(pred_df['reward_total']):.4f}")
print(f"Average BERTScore: {np.mean(pred_df['reward_bert']):.4f}")
print(f"Average LLM Score: {np.mean(pred_df['reward_llm']):.4f}")

### Сохранение результатов

In [ ]:
pred_df.to_csv(f'/content/drive/MyDrive/ПЗАД проект/Evaluation/PE_new_{HYPOTHESIS}.csv', index=False)
print(f"Результаты сохранены в PE_new_{HYPOTHESIS}.csv")
pred_df.head()